# 06 · Diagnóstico del colapso de MobileNetV2

MobileNetV2 colapsa al entrenar: la precisión de entrenamiento se queda en el azar
(0.015 contra 1/79 = 0.0127) mientras la `val_loss` explota a 84. Es reproducible en
CPU y en GPU, y ResNet50V2 falla igual.

Ya está descartado el backbone. Con el preprocess correcto, sus features varían
más entre imágenes que las de MobileNetV3Small, que sí entrena:

| | entrada | media | std | ceros | var entre imágenes |
|---|---|---|---|---|---|
| mobilenet_v2 | [-1, 1] | 0.1826 | 0.8054 | 91.9 % | 0.07496 |
| mobilenet_v3_small | [0, 255] | 0.4189 | 2.0425 | 57.9 % | 0.06265 |

Queda la cabeza. La brecha entre una `loss` de entrenamiento sana y una `val_loss`
de 84 es la firma de sus tres `BatchNormalization`: normalizan con estadísticas de
lote al entrenar y con medias móviles al validar, y si ambas divergen el modelo
predice bien en un modo y pésimo en el otro.

Este notebook entrena solo la cabeza sobre features congeladas y ya extraídas, así
que cada combinación cuesta segundos en vez de media hora. Tres arquitecturas por
tres cabezas: las dos que colapsan más un control que funciona.

No produce un run ni toca el registro.

In [ ]:
# En Kaggle clona el repo; en local no hace nada.
import os
import subprocess
import sys
from pathlib import Path

if Path("/kaggle").exists():
    from kaggle_secrets import UserSecretsClient

    # /tmp y no /kaggle/working: esa carpeta se guarda como salida del notebook
    CLON = "/tmp/ButterflyModeling"
    AYUDA = "!f() { echo username=x-access-token; echo password=$GITHUB_TOKEN; }; f"
    os.environ["GITHUB_TOKEN"] = UserSecretsClient().get_secret("GITHUB_TOKEN")
    if not Path(CLON).exists():
        subprocess.run(["git", "-c", f"credential.helper={AYUDA}", "clone",
                        "https://github.com/CCSandoval/ButterflyModeling.git", CLON], check=True)
    sys.path.insert(0, CLON)
    os.chdir(CLON)
    print("clon en", CLON, "·", subprocess.run(
        ["git", "log", "-1", "--format=%h %s"], cwd=CLON,
        capture_output=True, text=True).stdout.strip())

In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "corpus.json").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Raíz del proyecto:", ROOT)

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf

from scripts import corpus
from scripts.architectures import ARCHITECTURES, IMG_SIZE
from scripts.dataPipeline import loadSplit
from scripts.vizStyle import SERIE_1, SERIE_2, SERIE_3, applyStyle

# las dos que colapsan, más MobileNetV3Small de control
ARQUITECTURAS = ("mobilenet_v2", "resnet50_v2", "mobilenet_v3_small")

LOTE = 32
LOTES_TRAIN = 120        # ~3840 imágenes: suficientes para que 79 clases tengan ejemplos
LOTES_VAL = 32
EPOCAS = 20
TASA = 1e-4              # la misma de los notebooks de entrenamiento
SEMILLA = 42

tf.keras.utils.set_random_seed(SEMILLA)
corpus.materializar()

dsTrain, clases = loadSplit("dataset/train", IMG_SIZE, LOTE)
dsVal, _ = loadSplit("dataset/validate", IMG_SIZE, LOTE, shuffle=False)
NUM_CLASES = len(clases)
print(f"{NUM_CLASES} clases · {IMG_SIZE[0]}x{IMG_SIZE[1]}")

## 1 · Las features del backbone

Se confirma lo ya medido, ahora para las tres. `var_entre_imagenes` cerca de cero
significaría que el backbone devuelve lo mismo para cualquier mariposa; si es
apreciable, el problema no está aquí.

In [ ]:
def extractor(nombre):
    """Backbone congelado más el pooling, que es lo que la cabeza recibe."""
    clase, preprocess, _ = ARCHITECTURES[nombre]
    base = clase(weights="imagenet", include_top=False, input_shape=(*IMG_SIZE, 3))
    base.trainable = False
    return tf.keras.Sequential([base, tf.keras.layers.GlobalAveragePooling2D()]), preprocess


def extraer(modelo, preprocess, ds, lotes):
    X, Y = [], []
    for x, y in ds.take(lotes):
        X.append(modelo(preprocess(tf.cast(x, tf.float32)), training=False).numpy())
        Y.append(y.numpy())
    return np.concatenate(X), np.concatenate(Y)


features = {}
for nombre in ARQUITECTURAS:
    modelo, preprocess = extractor(nombre)
    Xtr, Ytr = extraer(modelo, preprocess, dsTrain, LOTES_TRAIN)
    Xva, Yva = extraer(modelo, preprocess, dsVal, LOTES_VAL)
    features[nombre] = (Xtr, Ytr, Xva, Yva)
    print(f"{nombre:20s} dim={Xtr.shape[1]:5d}  media={Xtr.mean():7.4f}  std={Xtr.std():7.4f}  "
          f"ceros={100 * (Xtr == 0).mean():5.1f}%  var_entre_imagenes={Xtr.mean(axis=1).std():.5f}  "
          f"clases_con_ejemplos={int(Ytr.sum(axis=0).astype(bool).sum())}/{NUM_CLASES}")
    del modelo

## 2 · Las cabezas

`actual` es la que llevan hoy los nueve candidatos. `sin_bn` quita las
`BatchNormalization` conservando profundidad, y `minima` es el clasificador lineal
que la práctica habitual de transferencia pondría. Si `minima` y `sin_bn` aprenden
donde `actual` no, el culpable son las normalizaciones y no el tamaño.

In [ ]:
D = tf.keras.layers.Dense
BN = tf.keras.layers.BatchNormalization
Drop = tf.keras.layers.Dropout


def cabezas(numClases):
    return {
        "minima": [D(numClases, activation="softmax")],
        "sin_bn": [D(512, activation="relu"), Drop(0.30),
                   D(256, activation="relu"), Drop(0.40),
                   D(128, activation="relu"), Drop(0.50),
                   D(numClases, activation="softmax")],
        "actual": [D(512, activation="relu"), BN(), Drop(0.30),
                   D(256, activation="relu"), BN(), Drop(0.40),
                   D(128, activation="relu"), BN(), Drop(0.50),
                   D(numClases, activation="softmax")],
    }


filas = []
for arquitectura in ARQUITECTURAS:
    Xtr, Ytr, Xva, Yva = features[arquitectura]
    for nombre, capas in cabezas(NUM_CLASES).items():
        tf.keras.utils.set_random_seed(SEMILLA)
        modelo = tf.keras.Sequential(capas)
        modelo.compile(optimizer=tf.keras.optimizers.Adam(TASA),
                       loss="categorical_crossentropy",
                       metrics=["accuracy"])
        h = modelo.fit(Xtr, Ytr, validation_data=(Xva, Yva), epochs=EPOCAS,
                       batch_size=LOTE, verbose=0).history
        filas.append({
            "arquitectura": arquitectura,
            "cabeza": nombre,
            "train_acc": h["accuracy"][-1],
            "val_acc": h["val_accuracy"][-1],
            "val_loss": h["val_loss"][-1],
            "val_loss_max": max(h["val_loss"]),
        })
        print(f"{arquitectura:20s} {nombre:7s} train={h['accuracy'][-1]:.4f} "
              f"val={h['val_accuracy'][-1]:.4f} val_loss={h['val_loss'][-1]:8.3f} "
              f"pico={max(h['val_loss']):9.2f}")

resultados = pd.DataFrame(filas)
resultados

## 3 · Lectura

El azar está en 1/79 = 0.0127. Una cabeza que «aprende» se despega de ahí en
entrenamiento; el pico de `val_loss` delata la divergencia entre modos.

In [ ]:
applyStyle()
import matplotlib.pyplot as plt

AZAR = 1.0 / NUM_CLASES
colores = {"minima": SERIE_3, "sin_bn": SERIE_1, "actual": SERIE_2}

fig, ejes = plt.subplots(1, 2, figsize=(11, 4.5))
tabla = resultados.pivot(index="arquitectura", columns="cabeza", values="val_acc")
tabla = tabla.reindex(ARQUITECTURAS)[list(colores)]
tabla.plot.bar(ax=ejes[0], color=[colores[c] for c in tabla.columns], width=0.75, rot=15)
ejes[0].axhline(AZAR, color="#8a8880", linestyle="--", linewidth=1.2)
ejes[0].text(-0.45, AZAR, " azar", fontsize=8, va="bottom", color="#8a8880")
ejes[0].set(ylabel="accuracy (validación)", xlabel="", title="Qué cabeza aprende")

pico = resultados.pivot(index="arquitectura", columns="cabeza", values="val_loss_max")
pico = pico.reindex(ARQUITECTURAS)[list(colores)]
pico.plot.bar(ax=ejes[1], color=[colores[c] for c in pico.columns], width=0.75, rot=15,
              logy=True)
ejes[1].set(ylabel="pico de val_loss (log)", xlabel="", title="Divergencia entre modos")
fig.tight_layout()
plt.show()

print()
for arquitectura in ARQUITECTURAS:
    sub = resultados[resultados.arquitectura == arquitectura].set_index("cabeza")
    aprenden = [c for c in sub.index if sub.loc[c, "train_acc"] > 5 * AZAR]
    print(f"{arquitectura:20s} aprenden: {aprenden or 'ninguna'}")

## Conclusión

Tres lecturas posibles:

- **`minima` y `sin_bn` aprenden, `actual` no** — son las `BatchNormalization` de la
  cabeza. Afecta a las nueve arquitecturas como riesgo latente y arreglarlo sube
  todas las bases, no solo rescata a MobileNetV2.
- **Ninguna aprende sobre MobileNetV2 pero sí sobre el control** — el problema está
  en esas features para esta tarea, y MobileNetV2 se documenta como candidato
  descartado.
- **Las tres aprenden** — ni backbone ni cabeza: hay que mirar el camino completo de
  entrenamiento, o sea augmentation, pesos de clase y la fase de descongelado.